# 32 — AD Through Calibration

Differentiate **through** the calibration loop: model prices → optimizer → calibrated parameters → downstream Greeks.

This is impossible with bump-and-reprice but trivial with JAX AD.

**Pipeline**: market data → calibration → calibrated params → exotic price → ∂(exotic) / ∂(market data)

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.engines.analytic.barrier import barrier_price

---
## 1. Generate Synthetic Market Data

In [ ]:
S, r, q = 100.0, 0.03, 0.01
# "True" Heston params
TRUE_V0, TRUE_KAPPA, TRUE_THETA, TRUE_XI, TRUE_RHO = 0.04, 1.5, 0.04, 0.3, -0.7

# Market instruments: option prices at various (K, T)
strikes = jnp.array([85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0])
maturities = jnp.array([0.25, 0.5, 1.0])

# Generate "market" prices
market_prices = []
for T in maturities:
    for K in strikes:
        p = heston_price(S, K, float(T), r, q, TRUE_V0, TRUE_KAPPA, TRUE_THETA, TRUE_XI, TRUE_RHO, 1)
        market_prices.append(float(p))
market_prices = jnp.array(market_prices)

print(f"Market instruments: {len(market_prices)} vanilla options")
print(f"Strike range: [{float(strikes[0])}, {float(strikes[-1])}]")
print(f"Maturity range: [{float(maturities[0])}, {float(maturities[-1])}]")

---
## 2. Differentiable Calibration Loop

In [ ]:
def model_prices(params):
    """Compute model prices for all instruments given Heston params."""
    v0, kappa, theta, xi, rho = params[0], params[1], params[2], params[3], params[4]
    prices = []
    for T in maturities:
        for K in strikes:
            p = heston_price(S, K, float(T), r, q, v0, kappa, theta, xi, rho, 1)
            prices.append(p)
    return jnp.array(prices)

def loss(params, target_prices):
    """Sum of squared pricing errors."""
    model = model_prices(params)
    return jnp.sum((model - target_prices) ** 2)

# Gradient descent calibration (differentiable!)
def calibrate(target_prices, n_steps=200, lr=1e-4):
    # Initial guess
    params = jnp.array([0.05, 2.0, 0.05, 0.4, -0.5])
    
    grad_loss = jax.grad(loss)
    
    def step(params, _):
        g = grad_loss(params, target_prices)
        # Clip gradients for stability
        g = jnp.clip(g, -1.0, 1.0)
        new_params = params - lr * g
        # Project to valid domain
        new_params = new_params.at[0].set(jnp.clip(new_params[0], 0.001, 1.0))  # v0 > 0
        new_params = new_params.at[1].set(jnp.clip(new_params[1], 0.01, 10.0))  # kappa > 0
        new_params = new_params.at[2].set(jnp.clip(new_params[2], 0.001, 1.0))  # theta > 0
        new_params = new_params.at[3].set(jnp.clip(new_params[3], 0.01, 2.0))   # xi > 0
        new_params = new_params.at[4].set(jnp.clip(new_params[4], -0.99, 0.99)) # |rho| < 1
        return new_params, loss(new_params, target_prices)
    
    params, losses = jax.lax.scan(step, params, jnp.arange(n_steps))
    return params, losses

calibrated, losses = calibrate(market_prices)

print(f"Calibrated vs True:")
names = ['v0', 'kappa', 'theta', 'xi', 'rho']
true_vals = [TRUE_V0, TRUE_KAPPA, TRUE_THETA, TRUE_XI, TRUE_RHO]
for n, t, c in zip(names, true_vals, calibrated):
    print(f"  {n:6s}: true={t:.4f}  calibrated={float(c):.4f}")

plt.figure(figsize=(8, 4))
plt.semilogy(np.array(losses))
plt.xlabel('Iteration')
plt.ylabel('Loss (SSE)')
plt.title('Calibration Convergence')
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. End-to-End: ∂(Exotic Price)/∂(Market Data)

In [ ]:
def end_to_end(target_prices):
    """Market data → calibration → exotic price."""
    params, _ = calibrate(target_prices, n_steps=100)
    v0, kappa, theta, xi, rho = params
    # Price an exotic: OTM put
    exotic = heston_price(S, 85.0, 2.0, r, q, v0, kappa, theta, xi, rho, -1)
    return exotic

# Jacobian: ∂(exotic price) / ∂(each market vanilla price)
J = jax.jacobian(end_to_end)(market_prices)

print(f"Jacobian shape: {J.shape}  # ∂(exotic) / ∂(21 market prices)")

# Reshape to (T, K) grid
J_grid = np.array(J).reshape(len(maturities), len(strikes))

plt.figure(figsize=(9, 5))
for i, T in enumerate(maturities):
    plt.plot(np.array(strikes), J_grid[i], 'o-', label=f'T={float(T):.2f}')
plt.xlabel('Strike of Market Instrument')
plt.ylabel('∂(Exotic Price) / ∂(Market Price)')
plt.title('Exotic Sensitivity to Each Calibration Instrument')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. ∂(Calibrated Params)/∂(Market Data)

In [ ]:
def calib_params_only(target_prices):
    params, _ = calibrate(target_prices, n_steps=100)
    return params

J_params = jax.jacobian(calib_params_only)(market_prices)
print(f"Jacobian shape: {J_params.shape}  # (5 params, 21 market prices)")

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (ax, name) in enumerate(zip(axes, names)):
    row = np.array(J_params[i]).reshape(len(maturities), len(strikes))
    im = ax.imshow(row, cmap='RdBu_r', aspect='auto')
    ax.set_title(f'∂{name}/∂market')
    ax.set_xticks(range(len(strikes)))
    ax.set_xticklabels([f'{int(k)}' for k in strikes], fontsize=7)
    ax.set_yticks(range(len(maturities)))
    ax.set_yticklabels([f'{float(t):.2f}' for t in maturities], fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.6)

fig.suptitle('Calibrated Parameter Sensitivity to Market Prices', fontsize=14)
fig.tight_layout()
plt.show()

---
## 5. Implicit Function Theorem Comparison

The IFT says: if $F(\theta^*, p) = 0$ at the optimum, then
$\frac{d\theta^*}{dp} = -\left(\frac{\partial F}{\partial \theta}\right)^{-1} \frac{\partial F}{\partial p}$

In [ ]:
# Optimality condition: gradient of loss = 0
def F(params, target_prices):
    return jax.grad(loss)(params, target_prices)

# IFT: dθ*/dp = -(∂F/∂θ)⁻¹ @ (∂F/∂p)
dF_dtheta = jax.jacobian(F, argnums=0)(calibrated, market_prices)
dF_dp = jax.jacobian(F, argnums=1)(calibrated, market_prices)

J_ift = -jnp.linalg.solve(dF_dtheta, dF_dp)

print(f"IFT Jacobian shape: {J_ift.shape}")
print(f"\nComparison (first 3 market instruments):")
print(f"  {'Param':>6}  {'AD':>10}  {'IFT':>10}")
for i, name in enumerate(names):
    ad_val = float(J_params[i, 0])
    ift_val = float(J_ift[i, 0])
    print(f"  {name:>6}  {ad_val:>10.6f}  {ift_val:>10.6f}")

---
## 6. Exercises

1. **Different exotic**: Replace the OTM put with a barrier option and compute end-to-end sensitivities.
2. **Yield curve calibration**: Differentiate through a piecewise yield curve bootstrap.
3. **Custom VJP**: Use `jax.custom_vjp` to implement implicit differentiation for the calibration.

---
*End of Notebook 32*